# Evaluación de entradas: Clasificación

Este notebook tiene como objetivo procesar y evaluar la entrada del usuario antes de que llegue al LLM o antes de ejecutar la siguiente parte de un sistema de IA.

## 1. Configuración

#### Carga de la clave API y las librerías de Python relevantes.

In [1]:
import os
import openai
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) #lectura en local del archivo .env

openai.api_key  = os.environ['OPENAI_API_KEY']

## 2. Función auxiliar

Implementamos la función auxiliar para enviar prompts al modelo y recibir respuestas directas ya vista en el notebook 'modelosDeLenguajeChatsYTokens'.

In [2]:
def get_completion_from_messages(messages, 
                                 model="gpt-3.5-turbo", 
                                 temperature=0, 
                                 max_tokens=500):
    response = openai.ChatCompletion.create(
        model=model,
        messages=messages,
        temperature=temperature, 
        max_tokens=max_tokens,
    )
    return response.choices[0].message["content"]

## 3. Evaluación y clasificación de las distintas peticiones

Clasificamos las peticiones o consultas del cliente para manejar diferentes casos. Para lograr esto usaremos un delimitador (delimiter), que viene a ser una forma de separar diferentes partes de una instrucción o de una salida, y ayuda al modelo a determinar cuáles son las diferentes secciones.

In [3]:
delimiter = "####"
system_message = f"""
Se te proporcionarán consultas de atención al cliente. \
La consulta de atención al cliente estará delimitada mediante los caracteres \
{delimiter} de delimitación.
Clasifica cada consulta en una categoría primaria \
y una categoría secundaria. 
Proporciona tu salida en formato JSON con las \
claves: primary y secondary.

Categorías primarias: Billing (facturación), Technical Support (soporte técnico), \
Account Management (gestión de cuentas) o General Inquiry (consulta general).

Categorías secundarias de Billing (facturación):
Cancelar suscripción o actualizar
Añadir un método de pago
Explicación de un cargo
Disputar un cargo

Categorías secundarias de Technical Support (soporte técnico):
Solución de problemas general
Compatibilidad de dispositivos
Actualizaciones de software

Categorías secundarias de Account Management (gestión de cuentas):
Restablecimiento de contraseña
Actualizar información personal
Cerrar cuenta
Seguridad de la cuenta

Categorías secundarias de General Inquiry (consulta general):
Información del producto
Precios
Comentarios
Hablar con una persona

"""
user_message = f"""\
Quiero que elimines mi perfil y todos mis datos de usuario"""
messages =  [  
{'role':'system', 
 'content': system_message},    
{'role':'user', 
 'content': f"{delimiter}{user_message}{delimiter}"},  
] 
response = get_completion_from_messages(messages)
print(response)

{
    "primary": "Account Management",
    "secondary": "Cerrar cuenta"
}


In [4]:
user_message = f"""\
Cuéntame más sobre vuestros televisores de pantalla plana"""
messages =  [  
{'role':'system', 
 'content': system_message},    
{'role':'user', 
 'content': f"{delimiter}{user_message}{delimiter}"},  
] 
response = get_completion_from_messages(messages)
print(response)

{
    "primary": "General Inquiry",
    "secondary": "Información del producto"
}


'primary' nos informa a que sección generica pertenece la entrada del usuario, y 'secondary' que es lo que quiere hacer exactamente para que el modelo pueda darle instrucciones más específicas sobre su caso.